In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

import os
import io
import av
from pathlib import Path

import h5py
import tifffile as tiff
import glob

import subprocess
import tempfile

## Folder Setup


In [4]:
FIJI_APP = "/Users/vuhepola/Desktop/Fiji"
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"

print(FIJI_APP)
print(ROOT)
print(DATA_DIR)

/Users/vuhepola/Desktop/Fiji
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly
/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data


## File Tracking


In [5]:
target_line = "SS04748"
target_line_dir =  DATA_DIR /  target_line

h5j_files = list(target_line_dir.glob(f"{target_line}*.h5j"))

info_arr = []

for f in h5j_files:
    file_name = f.name

    info = file_name.split("-")

    line = info[0]
    slide_code = info[1]
    gender = info[2]
    alignment = info[-1].split(".")[0]

    d = {
        "path":f.absolute(),
        "line":line,
        "slide_code":slide_code,
        "gender":gender,
        "alignment":alignment
    }

    print(d)
    
    info_arr.append(d)

{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I2-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j'), 'line': 'SS04748', 'slide_code': '20170324_20_I2', 'gender': 'f', 'alignment': 'unaligned_stack'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I4-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j'), 'line': 'SS04748', 'slide_code': '20170324_20_I4', 'gender': 'f', 'alignment': 'unaligned_stack'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I1-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j'), 'line': 'SS04748', 'slide_code': '20170324_20_I1', 'gender': 'f', 'alignment': 'unaligned_stack'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I3-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j'), 'line': 'SS04748', 'slide_code': '20170324_20_I3', 'ge

In [6]:
info_arr

[{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I2-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j'),
  'line': 'SS04748',
  'slide_code': '20170324_20_I2',
  'gender': 'f',
  'alignment': 'unaligned_stack'},
 {'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I4-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j'),
  'line': 'SS04748',
  'slide_code': '20170324_20_I4',
  'gender': 'f',
  'alignment': 'unaligned_stack'},
 {'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I1-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j'),
  'line': 'SS04748',
  'slide_code': '20170324_20_I1',
  'gender': 'f',
  'alignment': 'unaligned_stack'},
 {'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I3-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j'),
  'line': 'SS04748',
  

### Extract to Numpy Zip


In [ ]:
def extract_channel_to_array(h5j_path, channel_name):
    """Extracts H.265 byte stream, uses FFmpeg to unpack slices, and returns a 3D NumPy array."""
    print(f"Processing {channel_name}...")
    
    with h5py.File(h5j_path, 'r') as hf:
        dataset = hf[f"Channels/{channel_name}"]
        video_bytes = bytes(dataset[:])
        
    # Create a temporary directory for FFmpeg's intermediate files
    with tempfile.TemporaryDirectory() as tmpdir:
        h265_path = os.path.join(tmpdir, "stream.h265")
        slice_pattern = os.path.join(tmpdir, "slice_%03d.tif")
        
        # Write video payload to the temp folder
        with open(h265_path, "wb") as video_file:
            video_file.write(video_bytes)
            
        # FFmpeg to extract 2D slices
        subprocess.run([
            "ffmpeg", "-y", 
            "-i", h265_path, 
            "-pix_fmt", "gray16le", 
            slice_pattern
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # Grab all generated slices, sort them, and stack into a 3D array
        slice_files = sorted(glob.glob(os.path.join(tmpdir, "slice_*.tif")))
        slices = [tiff.imread(f) for f in slice_files]
        volume_3d = np.stack(slices, axis=0)
    
    print(f"Extracted {channel_name} into memory.")
    print(f"Volume Shape: {volume_3d.shape} | Data Type: {volume_3d.dtype}\n")
    
    return volume_3d

for file in info_arr:
    
    print(file)

    # Extract all 4 channels
    volume = []
    for i in range(4):
        channel_volume = extract_channel_to_array(
            h5j_path=file["path"], 
            channel_name=f"Channel_{i}"
        )
        volume.append(channel_volume)

    volume = np.array(volume)
    print(f"Raw stacked shape (C, Z, Y, X): {volume.shape}")

    # Reorder dimensions from (C, Z, Y, X) to (Z, C, Y, X)
    reordered = np.transpose(volume, (1, 0, 2, 3))
    print(f"Reordered 4D Volume Shape: {reordered.shape}")

    # Save directly to a compressed numpy zip file
    output_npz_path = DATA_DIR / file["line"] / f"{file["line"]}-{file["slide_code"]}.npz"
    np.savez_compressed(output_npz_path, image_data=reordered)
    print(f"\nPipeline Complete! Compressed NumPy archive saved to: {output_npz_path}")
    
    # break # If only one sample is needed

{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I2-f-20x-ventral_nerve_cord-Split_GAL4-unaligned_stack.h5j'), 'line': 'SS04748', 'slide_code': '20170324_20_I2', 'gender': 'f', 'alignment': 'unaligned_stack'}
Processing Channel_0...
Extracted Channel_0 into memory.
Volume Shape: (119, 1024, 1024) | Data Type: uint16

Processing Channel_1...
Extracted Channel_1 into memory.
Volume Shape: (119, 1024, 1024) | Data Type: uint16

Processing Channel_2...
Extracted Channel_2 into memory.
Volume Shape: (119, 1024, 1024) | Data Type: uint16

Processing Channel_3...
Extracted Channel_3 into memory.
Volume Shape: (119, 1024, 1024) | Data Type: uint16

Raw stacked shape (C, Z, Y, X): (4, 119, 1024, 1024)
Reordered 4D Volume Shape: (119, 4, 1024, 1024)

Pipeline Complete! Compressed NumPy archive saved to: /Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I2.npz


### Read from Numpy Zip


In [14]:
npz_files = list(target_line_dir.glob(f"{target_line}*.npz"))

npz_arr = []

for f in npz_files:
    file_name = f.name

    info = file_name.split("-")

    line = info[0]
    slide_code = info[1]

    d = {
        "path":f.absolute(),
        "line":line,
        "slide_code":slide_code,
    }

    print(d)
    
    npz_arr.append(d)

{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I3.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I3.npz'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I2.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I2.npz'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I1.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I1.npz'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I5.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I5.npz'}
{'path': PosixPath('/Users/vuhepola/GitHub/Repos/neuroinfo_fruitfly/data/SS04748/SS04748-20170324_20_I4.npz'), 'line': 'SS04748', 'slide_code': '20170324_20_I4.npz'}


## Load and Visualize


Load one file and visualize it with napari


In [15]:
npz_path = npz_arr[0]['path']


print("Loading data...")
with np.load(npz_path) as archive:
    print("Keys in archive:", archive.files)
    volume = archive['image_data']  # Shape: (127, 4, 1024, 1024)

print(volume.shape)

# Channels 0, 1, 2 are MCFO colors. Channel 3 is the Nc82 background.
signal = volume[:, 0:3, :, :]
reference = volume[:, 3, :, :]

# Normalize
# Convert to float32 and scale to [0, 1], 99.9th percentile
p99 = np.percentile(signal, 99.9)
signal_normalized = np.clip(signal.astype(np.float32) / p99, 0, 1)

print(f"Normalized Signal Shape: {signal_normalized.shape}")
print(f"Min: {signal_normalized.min()}, Max: {signal_normalized.max()}")

Loading data...
Keys in archive: ['image_data']
(129, 4, 1024, 1024)
Normalized Signal Shape: (129, 3, 1024, 1024)
Min: 0.0, Max: 1.0


In [ ]:
np_image_2 = np.transpose(volume, (0,2,3,1))

stack = np.asarray(np_image_2)
# stack = stack[..., 0]

print(stack.shape)

if stack.ndim == 4:
    n_channels = stack.shape[-1]
    if n_channels < 3:
        pad_width = [(0, 0)] * 3 + [(3 - n_channels, 0)]
        stack = np.pad(stack, pad_width, mode="constant")

    # n_channels in {3, 4} -> leave as-is (RGB / RGBA)
elif stack.ndim != 3:
    raise ValueError(f"Expected a (Z,Y,X) or (Z,Y,X,C) array, got {stack.shape}")

# Normalize to uint8 so matplotlib can display RGB/RGBA directly
# (and a grayscale stack stretched into a sensible range for cmap).
# p_lo, p_hi = np.percentile(stack, (1, 99.5))
# stack = np.clip((stack - p_lo) / max(p_hi - p_lo, 1e-9), 0, 1)
# stack = (stack * 255).astype(np.uint8)

n_slices = stack.shape[0]
is_rgb = stack.ndim == 4
imshow_kwargs = {} if is_rgb else {"cmap": "gray_r", "vmin": 0, "vmax": 255}

fig, ax = plt.subplots()
ax.axis("off")
im = ax.imshow(stack[0], **imshow_kwargs)
title = ax.set_title(f"Slice 1/{n_slices}")

def update(frame):
    im.set_data(stack[frame])
    title.set_text(f"Slice {frame + 1}/{n_slices}")
    return [im, title]

ani = animation.FuncAnimation(
    fig, update, frames=n_slices, interval=100, blit=False, repeat=True
)

plt.close(fig)
HTML(ani.to_jshtml())

(129, 1024, 1024, 4)
